In [1]:
import argparse
import os
import pickle

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
exp_name = 'collision-walking-rand'  # ckpt = 4000
ckpt = 200

action_scale = 1.0 # 動作のスケールを調整

In [4]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [5]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [6]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.01
# env_cfg['substeps'] = 10
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 100
env_cfg['termination_if_pitch_greater_than'] = 100

In [7]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [9]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

print("env_reset:", obs["policy"])

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
env_reset: tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
    

In [10]:
with torch.no_grad():
    
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    print("step :", cnt) 
    cnt += 1


Original actions :  tensor([[-0.1029,  0.0038,  0.8968, -0.2168, -0.4307, -0.8971,  0.3350,  0.0641,
          0.6694, -0.0840, -1.4133,  0.1074]], device='cuda:0')
Scaled actions :  tensor([[-0.1029,  0.0038,  0.8968, -0.2168, -0.4307, -0.8971,  0.3350,  0.0641,
          0.6694, -0.0840, -1.4133,  0.1074]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.5155e-16,
         -1.4749e-16,  1.3167e-04, -2.8193e-04, -1.3993e-03,  3.3722e-15,
          1.0089e-16, -1.1801e-16,  1.3167e-04, -2.8193e-04, -1.3993e-03,
          2.1418e-15,  1.0447e-14, -5.5904e-15,  1.1146e-03, -5.2326e-04,
         -5.5076e-02,  2.1188e-13,  7.2669e-15, -3.7366e-15,  1.1146e-03,
         -5.2326e-04, -5.5076e-02,  1.3460e-13, -1.0286e-01,  3.8334e-03,
          8.9682e-01, -2.1678e-01, -4.3067e-01, -8.9713e-01,  3.3501e-01,
          6.4110e-02,  6.6945e-01, -8.4019e-02, -1.4133e+00,  1.0743e-01]],
       device='cuda:0')
step : 0


In [11]:
with torch.no_grad(): 
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    print("step :", cnt)   
    cnt += 1


Original actions :  tensor([[ 5.3890e-03, -2.1604e-01,  1.2729e+00, -2.9854e-01, -1.2765e+00,
         -1.3262e+00,  7.3946e-01,  2.6616e-01,  9.8599e-01, -2.5108e-01,
         -2.5520e+00, -2.4452e-03]], device='cuda:0')
Scaled actions :  tensor([[ 5.3890e-03, -2.1604e-01,  1.2729e+00, -2.9854e-01, -1.2765e+00,
         -1.3262e+00,  7.3946e-01,  2.6616e-01,  9.8599e-01, -2.5108e-01,
         -2.5520e+00, -2.4452e-03]], device='cuda:0')
tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.2501e-04,
          7.4778e-05,  6.6665e-03, -3.7003e-03, -4.1116e-03, -9.7310e-03,
          2.7827e-03,  4.9182e-04,  5.2809e-03, -2.1411e-03, -1.8717e-02,
         -4.5941e-03,  2.1044e-02, -3.3657e-03,  4.2253e-02, -1.2123e-02,
         -6.5552e-02,  1.3525e-03, -1.2158e-02,  6.6770e-03,  3.0660e-02,
         -6.5215e-03, -4.0374e-02, -7.3103e-02,  5.3890e-03, -2.1604e-01,
          1.2729e+00, -2.9854e-01, -1.27

In [12]:
with torch.no_grad(): 
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    print("step :", cnt)   
    cnt += 1


Original actions :  tensor([[ 0.3697, -0.1504,  1.5118, -0.2812, -1.1997, -1.0916,  0.7902,  0.3256,
          0.8583, -0.0513, -2.3185,  0.0942]], device='cuda:0')
Scaled actions :  tensor([[ 0.3697, -0.1504,  1.5118, -0.2812, -1.1997, -1.0916,  0.7902,  0.3256,
          0.8583, -0.0513, -2.3185,  0.0942]], device='cuda:0')
tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  3.6586e-04,
         -1.8318e-03,  1.9289e-02, -8.6441e-03, -2.2839e-02, -2.2710e-02,
          9.5167e-03,  2.5868e-03,  1.4331e-02, -5.1203e-03, -4.1398e-02,
         -9.5966e-03,  2.9358e-02, -1.6348e-02,  9.6156e-02, -6.2413e-02,
         -2.9389e-01, -1.8855e-01,  7.9621e-03,  1.6180e-02,  4.5914e-02,
         -1.3621e-02, -6.8052e-02, -7.5582e-02,  3.6968e-01, -1.5041e-01,
          1.5118e+00, -2.8123e-01, -1.1997e+00, -1.0916e+00,  7.9020e-01,
          3.2563e-01,  8.5833e-01, -5.1256e-02, -2.3185e+00,  9.4154e-02]],
    

In [13]:
with torch.no_grad(): 
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    print("step :", cnt)   
    cnt += 1


Original actions :  tensor([[ 0.7587, -0.0393,  1.4971, -0.0922, -0.7005, -0.8354,  0.8167,  0.4667,
          0.7218,  0.2194, -1.5653,  0.1511]], device='cuda:0')
Scaled actions :  tensor([[ 0.7587, -0.0393,  1.4971, -0.0922, -0.7005, -0.8354,  0.8167,  0.4667,
          0.7218,  0.2194, -1.5653,  0.1511]], device='cuda:0')
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000, -1.0000,  1.0000,  0.0000,
          0.0000,  0.0083, -0.0038,  0.0331, -0.0119, -0.0372, -0.0316,  0.0161,
          0.0056,  0.0223, -0.0056, -0.0647, -0.0126,  0.0431, -0.0096,  0.0701,
         -0.0126, -0.1018,  0.0041,  0.0069,  0.0186,  0.0386, -0.0031, -0.0676,
         -0.0639,  0.7587, -0.0393,  1.4971, -0.0922, -0.7005, -0.8354,  0.8167,
          0.4667,  0.7218,  0.2194, -1.5653,  0.1511]], device='cuda:0')
step : 3


In [14]:
with torch.no_grad(): 
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    print("step :", cnt)   
    cnt += 1


Original actions :  tensor([[ 0.9505, -0.0018,  1.4155, -0.1731, -0.5094, -0.4942,  0.8736,  0.5363,
          0.5181,  0.2669, -1.2409,  0.1679]], device='cuda:0')
Scaled actions :  tensor([[ 0.9505, -0.0018,  1.4155, -0.1731, -0.5094, -0.4942,  0.8736,  0.5363,
          0.5181,  0.2669, -1.2409,  0.1679]], device='cuda:0')
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000, -1.0000,  1.0000,  0.0000,
          0.0000,  0.0147, -0.0042,  0.0464, -0.0127, -0.0480, -0.0409,  0.0224,
          0.0097,  0.0285, -0.0034, -0.0756, -0.0157,  0.0557, -0.0037,  0.0673,
         -0.0027, -0.1025,  0.0031,  0.0061,  0.0246,  0.0322,  0.0107, -0.0973,
         -0.0656,  0.9505, -0.0018,  1.4155, -0.1731, -0.5094, -0.4942,  0.8736,
          0.5363,  0.5181,  0.2669, -1.2409,  0.1679]], device='cuda:0')
step : 4


In [15]:
num_steps = 1000
for i in range(num_steps):
    with torch.no_grad():
        if i % 100 == 0:
            print("cnt :", cnt)   
        actions = policy(obs)

        # アクションに倍率を適用して動きを制限
        scaled_actions = actions * action_scale
        
        if i % 100 == 0:
            print("Original actions : ", actions)
            print("Scaled actions : ", scaled_actions)

        obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用

        cnt += 1

cnt : 5
Original actions :  tensor([[ 1.0938, -0.0272,  1.3408, -0.2647, -0.2428, -0.1670,  0.8942,  0.5720,
          0.3852,  0.2651, -0.9030,  0.1845]], device='cuda:0')
Scaled actions :  tensor([[ 1.0938, -0.0272,  1.3408, -0.2647, -0.2428, -0.1670,  0.8942,  0.5720,
          0.3852,  0.2651, -0.9030,  0.1845]], device='cuda:0')
cnt : 105
Original actions :  tensor([[-0.5666, -0.1378, -0.0059,  0.1428, -1.6689,  0.0536, -0.8831,  0.1622,
          0.9978, -0.3438, -0.3434,  0.3831]], device='cuda:0')
Scaled actions :  tensor([[-0.5666, -0.1378, -0.0059,  0.1428, -1.6689,  0.0536, -0.8831,  0.1622,
          0.9978, -0.3438, -0.3434,  0.3831]], device='cuda:0')
cnt : 205
Original actions :  tensor([[-3.5555e-01, -3.9864e-02, -6.1213e-02, -2.5368e-04, -9.0969e-01,
         -2.5124e-01, -1.2823e+00,  5.3345e-03,  1.2026e+00, -2.7201e-01,
          1.2092e-01,  2.5251e-01]], device='cuda:0')
Scaled actions :  tensor([[-3.5555e-01, -3.9864e-02, -6.1213e-02, -2.5368e-04, -9.0969e-01,
  

In [16]:
env.sim.stop()